# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/div828/Flyrank_starternotebook/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional lane: Lane 2 — Refresh / Content Opportunity Scoring.**

I'm picking this lane because it is the one lane where the starter repo already ships a full,
working, end-to-end example (`scripts/01`–`05`, `outputs/model_report.md`) that I can check my
own numbers against. The starter dataset has a clear, if imperfect, target
(`is_declining_label = trend_direction == "down"`), a documented baseline score, and a
documented model comparison (baseline rules vs. logistic regression vs. decision tree vs.
random forest, evaluated with client-holdout validation and precision@50). That gives me a
concrete floor to stand on in Week 1 and a concrete ceiling to try to beat over the next 7
weeks — I'm not framing a question in a vacuum, I'm framing it against evidence that a rule
already exists and can already be measured.

It also matches the shape of decision FlyRank's reviewers actually face: too many candidate
pages, not enough review time, and a real cost to reviewing the wrong ones first. Lanes 1
(signal analysis) and 3 (clustering) are more exploratory and don't end in an action queue;
Lane 4 (CTR scoring) is a good second choice but is really a special case of the same
"which page first" decision, just scoped to one failure mode (under-captured clicks). Lane 2
is the broadest version of the decision I want to study, and I can narrow into a CTR-flavored
sub-question later if the data pushes me that way.

I am **not** freestyling. I don't yet have a question that the four predefined lanes can't
already hold, and freestyle asks me to build my own joins/windows/labels from the 79M-row
warehouse with no starter pipeline to check against — that's a Week 4+ decision, not a Week 1
one, per the guide's "you can confirm or change your lane until the end of Week 4."

This is a provisional choice. I may narrow it (e.g., toward the CTR-review slice, or toward a
future-window decline/recovery label instead of the current-window proxy) once I've done the
signal audit in ML-04.

In [1]:
# Quick environment check for Section 1 -- confirm the starter file is where the lane guide says it is.
import pandas as pd

path = "../../data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(path)
print("Starter dataset loaded:", df.shape[0], "rows x", df.shape[1], "columns")
print("Distinct clients:", df["client_id"].nunique())


Starter dataset loaded: 30000 rows x 44 columns
Distinct clients: 32


## 2. The question: decision, action, cost of a wrong call

**Research question:** Given a client's inventory of existing content pages and their
trailing-90-day search and engagement signals, which pages should a content reviewer look at
**first** for refresh, expansion, protection, pruning, or monitoring?

**Unit of analysis:** one row = one pseudonymized content page (`content_id`), scored and
ranked within its client (`client_id`). This is the starter dataset's grain, and it's the
grain the decision actually operates at — a reviewer picks up one page at a time.

**Decision it improves:** which page(s) in a large, growing content inventory get a limited
reviewer's attention this week, instead of reviewers scanning the whole inventory in no
particular order or relying only on hand-built rules.

**Who acts, and what they do:** a content/SEO reviewer (or an editorial lead triaging a
queue) opens the top of a ranked list and either refreshes the page's content, expands thin
content, protects a page that's doing well from being deprioritized, prunes a page with no
real demand, or flags it to keep watching. The output is a ranked queue with a suggested
action and a short reason code per page, not a single number with no explanation.

**Model output:** for each page, a priority score (0–100) plus one or more reason codes
(e.g. `declining_with_demand`, `low_ctr_visible_page`) and a suggested action label. This
is a ranking/scoring output, not a single verdict.

**Cost of a wrong recommendation:**
- *False positive* (page ranked high but not actually worth reviewing): wastes a reviewer's
  time — the scarce resource in this whole problem — and erodes trust in the queue, so
  reviewers start ignoring it.
- *False negative* (a genuinely declining, high-demand page ranked low or missed): the page
  keeps losing visibility/clicks/sessions un-reviewed, which is a real, if hard to price
  precisely, opportunity cost — lost traffic and, downstream, lost conversions the page would
  otherwise have driven.
- Because reviewer time is limited and finite (a queue of, say, the top 20 or 50 pages), the
  practical cost is asymmetric: wasting a slot near the top of the queue is more costly than
  missing a candidate far down the list, which is why precision@K (not plain accuracy) is the
  right way to measure this later.

**Why data or ML can help at all (not just "train a model"):** a single hand-written
if-statement rule (the starter's `baseline_refresh_score`) already exists and is cheap to
compute, so the bar for ML is to show it beats that rule on a metric that matches the real
decision (precision@K), not just to exist. The reason a plain rule likely isn't enough is that
"worth reviewing" depends on several signals moving together and trading off against each
other — demand (impressions), position, trend, freshness, content depth, engagement — and the
right combination and weighting of those signals is exactly the kind of messy, multi-signal,
possibly non-linear pattern that a simple weighted rule cannot capture as well as a model that
can learn interactions from data. Whether that's actually true for my data is an empirical
question I'll test in ML-06/07/08, not an assumption I'm allowed to skip.

In [2]:
# Section 2 supporting check: how big is the review-capacity problem in practice?
# If the pool of "plausible candidates" is small, a human could just look at all of them --
# no ranking problem exists. If it's large, ranking/prioritization has real value.

visible_pages = (df["impressions_90d"] >= 500).sum()
declining_pages = (df["trend_direction"] == "down").sum()

print(f"Pages with >=500 impressions in the last 90 days (a plausible 'worth reviewing' floor): "
      f"{visible_pages:,} of {len(df):,} ({visible_pages/len(df):.1%})")
print(f"Pages currently labeled 'declining' (trend_direction == 'down'): "
      f"{declining_pages:,} of {len(df):,} ({declining_pages/len(df):.1%})")


Pages with >=500 impressions in the last 90 days (a plausible 'worth reviewing' floor): 16,726 of 30,000 (55.8%)
Pages currently labeled 'declining' (trend_direction == 'down'): 16,262 of 30,000 (54.2%)


## 3. Quick look at the data (2-3 real numbers)

I loaded `data/raw/content_refresh_anonymized.csv` (30,000 rows x 44 columns, 32 clients) and
computed the numbers below directly from it, using the exact reason-code rules documented in
the lane guide (Section 5) so they're comparable to the starter pipeline's own numbers.

1. **16,262 of 30,000 pages (54.2%)** are currently labeled `trend_direction == "down"`.
   That's over half the inventory — far too many for a reviewer to work through by hand,
   which is the core reason a ranking/scoring lane (not a manual scan) is worth building.
2. **9,759 pages (32.5%)** meet the `low_ctr_visible_page` reason code (>=500 impressions,
   position 1-20, CTR < 0.5%) — a large, concrete, actionable segment where a title/meta
   review could plausibly help, distinct from the general "declining" bucket.
3. **7,076 pages (23.6%)** meet `page_one_decay_risk` (ranked in the top 10, but at least
   180 days old) — a segment worth protecting, not just fixing, which is exactly the kind of
   nuance a single "declining = bad" rule collapses away.
4. The starter pipeline's own documented model comparison (`outputs/model_report.md`,
   computed with client-holdout validation on this same file) shows the rule-based baseline
   reaches **precision@50 = 0.240** (about 12 of its top 50 picks correct) while a random
   forest reaches **precision@50 = 0.740** (about 37 of 50 correct) on the
   `is_declining_label` target. I did not re-derive this model result myself in this
   notebook — I'm citing it as committed, already-verified evidence that a learned ranking
   can beat a hand-written rule on this kind of data, which is exactly the bet Lane 2 asks me
   to test properly on my own label definition.

Together these numbers say: the candidate pool is large (>50% of the inventory by one simple
rule), it splits into distinguishable, differently-actionable segments (CTR problem vs. decay
risk vs. plain decline), and there's already documented evidence that ranking beats
rule-of-thumb scoring here. That's what makes Lane 2 look worth the next 7 weeks, rather than
a lane I'm choosing by default.

In [3]:
# Section 3: the 2-3 real numbers, computed directly (not copied from the report).

n = len(df)

declining = (df["trend_direction"] == "down").sum()

low_ctr_visible_page = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0) & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
).sum()

page_one_decay_risk = (
    (df["avg_position"] > 0) & (df["avg_position"] <= 10)
    & (df["content_age_days"] >= 180)
).sum()

print(f"1) Declining pages (trend_direction == 'down'): {declining:,} / {n:,} = {declining/n:.1%}")
print(f"2) 'low_ctr_visible_page' segment (>=500 impr, pos 1-20, CTR<0.5%): "
      f"{low_ctr_visible_page:,} / {n:,} = {low_ctr_visible_page/n:.1%}")
print(f"3) 'page_one_decay_risk' segment (pos<=10, age>=180d): "
      f"{page_one_decay_risk:,} / {n:,} = {page_one_decay_risk/n:.1%}")

# Reference only -- not recomputed here, cited from the repo's own committed, verified output.
print()
print("Reference (from outputs/model_report.md, already verified, not recomputed in this notebook):")
print("  baseline_rules  precision@50 = 0.240")
print("  random_forest   precision@50 = 0.740")


1) Declining pages (trend_direction == 'down'): 16,262 / 30,000 = 54.2%
2) 'low_ctr_visible_page' segment (>=500 impr, pos 1-20, CTR<0.5%): 9,759 / 30,000 = 32.5%
3) 'page_one_decay_risk' segment (pos<=10, age>=180d): 7,076 / 30,000 = 23.6%

Reference (from outputs/model_report.md, already verified, not recomputed in this notebook):
  baseline_rules  precision@50 = 0.240
  random_forest   precision@50 = 0.740


## 4. Careful words: what I can and can't claim

**What I can claim, once this lane is built out:**
- *Observed*: which signals (impressions, position, CTR, freshness, word count, engagement)
  co-occur with pages currently labeled declining, low-CTR, or decaying, in this anonymized
  90-day snapshot.
- *Directional*: that a learned ranking, properly validated with a client-holdout split,
  orders pages by review-worthiness better than a fixed hand-written rule, measured by
  precision@K against a defined label.
- *Decision-support*: a ranked queue with reason codes that a human reviewer can inspect,
  agree or disagree with, and use to spend limited review time better than reviewing in no
  particular order.

**What this work will never claim:**
- That refreshing a page **causes** a recovery. I have no experiment and no causal design —
  at most I can say a page looks like a promising *candidate* for review, not that fixing it
  will work.
- That I've reverse-engineered or proven anything about Google's ranking algorithm, or about
  what any AI platform "understands" about a page.
- That the current-window label I start from (`trend_direction == "down"`, borrowed from the
  starter pipeline) is the ideal target. The lane guide flags this explicitly as a **proxy
  label** — it's calculated from the same window as the features, not a future outcome — so
  I'll treat any result built on it as a beginner baseline to improve on, and I intend to move
  toward a genuine past-window-features -> future-window-outcome label as the internship
  progresses, exactly as the guide recommends.
- That a high score guarantees anything about an individual page. Precision@K is a statement
  about the ranking as a whole, checked out-of-sample; it says little about any single row.
- I will not use or imply any client name, URL, domain, or raw query anywhere — the data is
  pseudonymized on purpose and I'll keep it that way in every notebook and in the final
  public write-up.

In [4]:
# Section 4 supporting check: confirm the label I'll start from really is a same-window
# proxy (not a future outcome), so my caveats above are grounded in the data, not asserted.

print("trend_direction value counts (this IS the source of is_declining_label):")
print(df["trend_direction"].value_counts())
print()
print("Rows with avg_position == 0 (means 'no data', not 'rank zero'):",
      (df["avg_position"] == 0).sum())


trend_direction value counts (this IS the source of is_declining_label):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Rows with avg_position == 0 (means 'no data', not 'rank zero'): 1205


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.